In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/sp-estimation

Mounted at /content/drive
/content/drive/MyDrive/sp-estimation


In [ ]:
!pip install -q sentence-transformers pyyaml reportlab scipy
!python -c "import torch; print('GPU available:', torch.cuda.is_available())"
!python -m src.check_pairs | tail -3

GPU available: True

All 78 names resolved.
Pairs that would be skipped for being too small: none


In [ ]:
!python -m src.run_experiment --pairs 1 2 --modes combined --set embeddings.cache_dir=/content/cache

device=cuda  backend=hf  pairs=2
config.json: 100% 570/570 [00:00<00:00, 2.63MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 302kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 10.1MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3.88MB/s]

model.safetensors: downloading bytes:  92% 404M/440M [00:01<00:00, 282MB/s, 36.2MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:04<00:00, 86.0MB/s, 37.7MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:04<00:00, 91.3MB/s, 39.2MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 6755.26it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTE

In [ ]:
!python -m src.run_experiment --modes combined --set embeddings.cache_dir=/content/cache

device=cuda  backend=hf  pairs=39
[pair 1] combined: already done, skipping (use --force to redo)
[pair 2] combined: already done, skipping (use --force to redo)
Loading weights: 100% 199/199 [00:00<00:00, 4653.06it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 103/103 [00:00<00:0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print(os.path.exists('/content/drive/MyDrive/sp-estimation/data/tawos_story_points.csv'))
print(os.listdir('/content/drive/MyDrive/sp-estimation/results/combined/pairs')[:5])

Mounted at /content/drive
True
['pair_01.json', 'pair_02.json', 'pair_03.json', 'pair_04.json', 'pair_05.json']


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/sp-estimation/data/tawos_story_points.csv')
df = df[df.Story_Point > 0]
g = df.groupby('Project_Name').Story_Point
out = pd.DataFrame({'n': g.size(), 'median': g.median(), 'p99': g.quantile(.99),
                    'max': g.max(), 'n_over_100': g.apply(lambda s: (s > 100).sum())})
print(out.sort_values('max', ascending=False).to_string())

                                         n  median     p99         max  n_over_100
Project_Name                                                                      
Spring DataCass                        216     3.0   92.50  26710114.0           2
The Titanium SDK                      4422     5.0   21.00      6765.0          10
Atlassian Jira Cloud                   352     4.0    8.00      3030.0           1
Spring XD                             3542     3.0   20.00      2567.0           1
Atlassian Software Cloud               296     4.0   20.00       688.0           1
Atlassian Atlassian Software Server    326     3.0   20.00       688.0           1
Lsstcorp Data management             20065     3.0   82.68       300.0         119
Moodle                                1528     5.0  100.00       195.0           3
Appcelerator Daemon                    242     5.0   46.39       144.0           1
Appcelerator Studio                   3295     5.0   20.00       144.0           1
Tita

In [ ]:
!python -m src.run_experiment --modes bert sbert --set embeddings.cache_dir=/content/cache

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/sp-estimation
!pip install -q sentence-transformers pyyaml reportlab scipy
p='configs/default.yaml'
s=open(p).read()
if 'max_story_point' not in s:
    s=s.replace("  min_source_issues: 50","  max_story_point: 500\n  min_source_issues: 50",1)
    open(p,'w').write(s)
print(s[s.index('data:'):s.index('split:')])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/sp-estimation
data:
  csv_path: data/tawos_story_points.csv   # produced by src/export_tawos.py
  pairs_path: data/pairs.csv
  # Column names inside the CSV. Edit if your export uses other names.
  columns:
    id: Issue_ID
    project: Project_Name
    title: Title
    description: Description_Text
    created: Creation_Date
    story_point: Story_Point
  # Project names in pairs.csv are the short names from the assignment sheet.
  # If a name does not match a TAWOS project automatically, map it here:
  #   "Short name in pairs.csv": "Exact name in TAWOS"
  project_aliases:
    "C++ driver (MongoDB)": "The Mongo C++ driver"
    "Evergreen": "The MongoDB Engineering"
    "C++ driver (Appcelerator)": "Aptana Studio"
  max_story_point: 500
  min_source_issues: 50       # a pair is skipped (and reported) below this
  min_target_test_issues

In [ ]:
!python -m src.check_pairs | head -20
!python -m src.check_pairs | tail -3


CLEANING RULE: story point > 500 removed (7 issues):
                                     count         max
project                                               
Atlassian Atlassian Software Server      1       688.0
Atlassian Jira Cloud                     1      3030.0
Atlassian Software Cloud                 1       688.0
Spring DataCass                          2  26710114.0
Spring XD                                1      2567.0
The Titanium SDK                         1      6765.0

63003 usable issues in 39 projects (story point > 0, non-empty text)

ALL TAWOS PROJECTS WITH STORY POINTS (usable issues):
project
Lsstcorp Data management               20065
The MongoDB Engineering                 5069
The Titanium SDK                        4421
Mule                                    3694
Spring XD                               3541

All 78 names resolved.
Pairs that would be skipped for being too small: none


In [ ]:
import shutil
shutil.copytree('results','results_before_cap',dirs_exist_ok=True)
print('backed up')

backed up


In [ ]:
!python -m src.run_experiment --pairs 1 4 5 8 11 17 18 20 25 36 38 39 --modes combined --force --set embeddings.cache_dir=/content/cache

device=cuda  backend=hf  pairs=12
config.json: 100% 570/570 [00:00<00:00, 2.55MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 235kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 15.7MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 57.2MB/s]

model.safetensors: downloading bytes:  55% 243M/440M [00:01<00:00, 317MB/s, 19.0MB/s  ]
model.safetensors: reconstructing file:  15% 67.1M/440M [00:01<00:07, 50.7MB/s]
model.safetensors: downloading bytes:  68% 299M/440M [00:01<00:00, 245MB/s, 26.6MB/s  ]
model.safetensors: downloading bytes:  87% 383M/440M [00:01<00:00, 261MB/s, 32.2MB/s  ]
model.safetensors: reconstructing file:  76% 335M/440M [00:02<00:00, 210MB/s, 25.3MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 186MB/s, 37.0MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 198MB/s, 40.6MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 6847.65it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/sp-estimation
!pip install -q sentence-transformers pyyaml reportlab scipy
!python -m src.run_experiment --modes sbert --set embeddings.cache_dir=/content/cache

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/sp-estimation
device=cpu  backend=hf  pairs=39
[pair 1] sbert: already done, skipping (use --force to redo)
[pair 2] sbert: already done, skipping (use --force to redo)
[pair 3] sbert: already done, skipping (use --force to redo)
[pair 4] sbert: already done, skipping (use --force to redo)
[pair 5] sbert: already done, skipping (use --force to redo)
[pair 6] sbert: already done, skipping (use --force to redo)
[pair 7] sbert: already done, skipping (use --force to redo)
[pair 8] sbert: already done, skipping (use --force to redo)
[pair 9] sbert: already done, skipping (use --force to redo)
[pair 10] sbert: already done, skipping (use --force to redo)
[pair 11] sbert: already done, skipping (use --force to redo)
[pair 12] sbert: already done, skipping (use --force to redo)
[pair 13] sbert: already done, skipping (use --force to redo)
[pai

In [ ]:
import json, glob
def load(m): return {json.load(open(f))['pair_id']: json.load(open(f)) for f in glob.glob(f'/content/drive/MyDrive/sp-estimation/results/{m}/pairs/pair_*.json')}
c, s = load('combined'), load('sbert')
print(len(c), len(s))
bad = [p for p in s if (s[p]['n_source_dev'], s[p]['n_target_test']) != (c[p]['n_source_dev'], c[p]['n_target_test'])]
print('pairs whose split sizes differ between sbert and combined:', bad)
print('backends:', {r['backend'] for r in list(c.values()) + list(s.values())})

39 39
pairs whose split sizes differ between sbert and combined: []
backends: {'hf'}
